#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [1]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


#`Setup And Update ComfyUI`



In [6]:
from pathlib import Path
import os # Added for path existence check

OPTIONS = {}

DRIVE_PATH = "MyDrive"  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

# Attempt to use Google Drive path
temp_base_dir_drive = Path(f'/content/drive/{DRIVE_PATH}')
BASE_DIR = Path('/content') # Default to /content

# Check if the desired Google Drive path exists and is a directory
if DRIVE_PATH and temp_base_dir_drive.is_dir():
    BASE_DIR = temp_base_dir_drive
    print(f"Using Google Drive as base directory: {BASE_DIR}")
else:
    print(f"Google Drive path '{temp_base_dir_drive}' not found or not mounted. Using local /content as base directory.")
    BASE_DIR = Path('/content')


# Set the WORKSPACE to the full absolute path of the ComfyUI installation directory
WORKSPACE = BASE_DIR / 'ComfyUI'

# Change current directory to the base directory where ComfyUI should reside
# This is where git clone will be executed if ComfyUI is not found
%cd {BASE_DIR}

# Check if ComfyUI directory exists. If not, clone it.
if not WORKSPACE.is_dir():
    print("-= Initial setup ComfyUI =-")
    # Clone ComfyUI into the specified WORKSPACE directory name within BASE_DIR
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE.name}

# Now change into the ComfyUI WORKSPACE for further operations
# Ensure WORKSPACE actually exists before changing into it
if not WORKSPACE.is_dir():
    print(f"Error: ComfyUI directory not found at {WORKSPACE}. Please check cloning process.")
    raise FileNotFoundError(f"ComfyUI directory not found at {WORKSPACE}")

%cd {WORKSPACE}

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

print("\nComfyUI base setup complete. To generate AI videos, you will typically need to install specific custom nodes (e.g., AnimateDiff) and corresponding models. Please proceed to the 'Models Download' section to get started, or let me know if you need help finding specific custom nodes for video generation.")


Google Drive path '/content/drive/MyDrive' not found or not mounted. Using local /content as base directory.
/content
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 MB 8.3 MB/s eta 0:00:00


# `Models Download`

## Download MODELS

In [18]:
import os
from pathlib import Path
import shutil # Added for rmtree
import subprocess # Added for more robust command execution
import time # Added for sleep in retry logic

# WORKSPACE is expected to be defined in a previous cell (Oab-QzpVYnx_).
# Ensure aria2 is installed (it already was)

from google.colab import userdata

CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
print("Loaded API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")

# --- Define paths and ensure directories exist ---
# Change current directory to WORKSPACE for consistency in relative paths
os.chdir(WORKSPACE)
print(f"Current working directory set to: {os.getcwd()}")

CUSTOM_NODES_PATH = WORKSPACE / 'custom_nodes'
CHECKPOINTS_PATH = WORKSPACE / 'models' / 'checkpoints'
VAE_PATH = WORKSPACE / 'models' / 'vae'

CUSTOM_NODES_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)
VAE_PATH.mkdir(parents=True, exist_ok=True)

# Function to install custom nodes
def install_custom_node(url):
  repo_name = Path(url).stem
  target_dir = CUSTOM_NODES_PATH / repo_name
  print(f"Attempting to clone {url} into {target_dir}")

  # Always remove the directory if it exists to ensure a clean clone or re-clone
  if target_dir.is_dir():
      print(f"Removing existing directory {target_dir} for a clean install.")
      try:
          shutil.rmtree(target_dir)
      except OSError as e:
          print(f"Error removing directory {target_dir}: {e}. Cannot proceed with clean clone.")
          print(f"--- Contents of {CUSTOM_NODES_PATH} after {repo_name} attempt --- ")
          !ls -al {CUSTOM_NODES_PATH}
          print("--- End contents ---")
          return # Abort if we can't clean up

  CUSTOM_NODES_PATH.mkdir(parents=True, exist_ok=True)

  max_retries = 3
  for attempt in range(max_retries):
      print(f"Attempt {attempt + 1}/{max_retries} to clone {repo_name}...")
      try:
          # Set GIT_ASKPASS=/bin/true to prevent any interactive prompts
          env_vars = os.environ.copy()
          env_vars['GIT_ASKPASS'] = '/bin/true'
          print(f"Using environment variable GIT_ASKPASS={env_vars['GIT_ASKPASS']}")
          command = ["git", "clone", url, str(target_dir)]
          print(f"Executing: {' '.join(command)}")

          process = subprocess.run(command, capture_output=True, text=True, check=False, env=env_vars)

          if process.returncode == 0:
              print(f"Successfully cloned {repo_name} to {target_dir}.")
              break # Success, exit retry loop
          else:
              print(f"Error: git clone failed with exit code {process.returncode} for {repo_name} on attempt {attempt + 1}.")
              print("Git stdout:")
              print(process.stdout)
              print("Git stderr:")
              print(process.stderr)
              if "fatal: could not read Username" in process.stderr:
                  print("This error often indicates git attempting to prompt for credentials unexpectedly, or a network issue.")
              if attempt < max_retries - 1:
                  print("Retrying in 5 seconds...")
                  time.sleep(5)
              else:
                  print(f"Failed to clone {repo_name} after {max_retries} attempts.")
                  if target_dir.is_dir() and len(os.listdir(target_dir)) > 0:
                      print(f"Warning: Partial clone of {repo_name} detected at {target_dir}.")
      except Exception as e:
          print(f"An unexpected exception occurred during git clone for {repo_name}: {e}")
          if attempt < max_retries - 1:
              print("Retrying in 5 seconds...")
              time.sleep(5)
          else:
              print(f"Failed to clone {repo_name} after {max_retries} attempts due to an exception.")

  print(f"--- Contents of {CUSTOM_NODES_PATH} after {repo_name} attempt --- ")
  !ls -al {CUSTOM_NODES_PATH}
  print("--- End contents ---")

# Function to download models
def downloadModel(url, destination_folder, filename=None):
  # Ensure the destination folder exists (already handled above, but harmless here)
  Path(destination_folder).mkdir(parents=True, exist_ok=True)

  # Determine filename if not provided
  if filename is None:
    if 'huggingface.co' in url:
      filename = url.split('/')[-1].removesuffix('?download=true')
    else: # Civitai - derive from URL or default
      # For Civitai, the URL often ends with a model ID.
      # We'll use the URL's last segment (model ID) for now, but explicit filename is better.
      filename_from_url = Path(url).name
      # Assuming a common extension like .safetensors if not explicitly named for Civitai
      filename = f"{filename_from_url}.safetensors" if '.' not in filename_from_url else filename_from_url
      print(f"Warning: Filename not explicitly provided for Civitai URL. Using inferred filename: {filename}")

  # Check if file already exists before downloading
  destination_file_path = Path(destination_folder) / filename
  if destination_file_path.is_file():
    print(f"File {filename} already exists at {destination_folder}, skipping download.")
  else:
    print(f"Downloading {filename} to {destination_folder}")
    if 'huggingface.co' in url:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url} -d {destination_folder} -o {filename}
    else:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN} -d {destination_folder} -o {filename}


# ---- Install custom nodes ----
print("\n-= Installing Custom Nodes =-")

install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
install_custom_node('https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved') # Corrected URL

# ---- Download checkpoint models ----
print("\n-= Downloading Checkpoint Models =-")
# CyberRealistic Pony
# Explicitly providing filename for better control.
downloadModel('https://civitai.com/api/download/models/2071650', CHECKPOINTS_PATH, filename='cyberrealisticPony_v127Alt.safetensors')

# SD1.5
downloadModel('https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt', CHECKPOINTS_PATH, filename='v1-5-pruned-emaonly.ckpt')

# Some SD1.5 anime style
downloadModel('https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors', CHECKPOINTS_PATH, filename='AbyssOrangeMix2_hard.safetensors')

# ---- Download VAE models ----
print("\n-= Downloading VAE Models =-")
# VAE
downloadModel('https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors', VAE_PATH, filename='vae-ft-mse-840000-ema-pruned.safetensors')

print("\nAll downloads and installations attempted. Please check the output for any errors, especially regarding 'ComfyUI-AnimateDiff-Evolved' and confirm models are in the correct ComfyUI directories.")

# Ensure we end up in WORKSPACE
os.chdir(WORKSPACE)

Loaded API key: ✅
Current working directory set to: /content/ComfyUI

-= Installing Custom Nodes =-
Attempting to clone https://github.com/ltdrdata/ComfyUI-Manager.git into /content/ComfyUI/custom_nodes/ComfyUI-Manager
Removing existing directory /content/ComfyUI/custom_nodes/ComfyUI-Manager for a clean install.
Attempt 1/3 to clone ComfyUI-Manager...
Using environment variable GIT_ASKPASS=/bin/true
Executing: git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
Successfully cloned ComfyUI-Manager to /content/ComfyUI/custom_nodes/ComfyUI-Manager.
--- Contents of /content/ComfyUI/custom_nodes after ComfyUI-Manager attempt --- 
total 24
drwxr-xr-x  3 root root 4096 Sep 21 09:54 .
drwxr-xr-x 24 root root 4096 Sep 21 09:32 ..
drwxr-xr-x 14 root root 4096 Sep 21 09:55 ComfyUI-Manager
-rw-r--r--  1 root root 5151 Sep 21 09:32 example_node.py.example
-rw-r--r--  1 root root 1301 Sep 21 09:32 websocket_image_save.py
--- End contents ---
Attemp

## Start ComfyUI Server

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies.
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-09-21 09:56:07.088
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]   14.5 seconds: /content/ComfyUI/custom_

Exception in thread Thread-3 (iframe_thread):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1120/133609189.py", line 17, in iframe_thread
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  File "/usr/lib/python3.13/subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process


ComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)

FETCH ComfyRegistry Data [DONE]
[INFO] [ComfyUI-Manager] default cache updated: https://api.comfy.org/nodes
FETCH DATA from: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/custom-node-list.json [DONE]
[INFO] [ComfyUI-Manager] All startup tasks have been completed.


## OTHERS

In [8]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

--2026-09-21 09:37:03--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt [following]
--2026-09-21 09:37:03--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/773d9e5ddbf6a84612066b3ea00a0bcfe4ba870a9befc88816f71b5927e76e38?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27v1-5-pruned-emaonly.ckpt%3B+filename%3D%22v1-5-pruned-emaonly.ckpt%22%3B&X-Xet-Cas-Uid=public&user_id=public&E

## LIST MODELS

In [ ]:
!ls -al ./models/checkpoints/

total 6775444
drwxr-xr-x  2 root root       4096 Aug 30 23:40 .
drwxr-xr-x 22 root root       4096 Aug 30 23:39 ..
-rw-r--r--  1 root root 6938040682 Aug 30 23:40 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Aug 30 23:39 put_checkpoints_here


### Verify Models and Custom Nodes

In [9]:
# List contents of the checkpoints directory
print('Contents of ./models/checkpoints/')
!ls -al {WORKSPACE}/models/checkpoints/

Contents of ./models/checkpoints/
total 16381104
drwxr-xr-x  2 root root       4096 Sep 21 09:37 .
drwxr-xr-x 28 root root       4096 Sep 21 09:32 ..
-rw-r--r--  1 root root 5570803627 Sep 21 09:38 AbyssOrangeMix2_hard.safetensors
-rw-r--r--  1 root root 6938040682 Sep 21 09:35 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Sep 21 09:32 put_checkpoints_here
-rw-r--r--  1 root root 4265380512 Sep 21 09:37 v1-5-pruned-emaonly.ckpt


In [10]:
# List contents of the VAE directory
print('\nContents of ./models/vae/')
!ls -al {WORKSPACE}/models/vae/


Contents of ./models/vae/
total 326808
drwxr-xr-x  2 root root      4096 Sep 21 09:38 .
drwxr-xr-x 28 root root      4096 Sep 21 09:32 ..
-rw-r--r--  1 root root         0 Sep 21 09:32 put_vae_here
-rw-r--r--  1 root root 334641190 Sep 21 09:38 vae-ft-mse-840000-ema-pruned.safetensors


In [11]:
# List contents of the custom_nodes directory
print('\nContents of ./custom_nodes/')
!ls -al {WORKSPACE}/custom_nodes/


Contents of ./custom_nodes/
total 20
drwxr-xr-x  2 root root 4096 Sep 21 09:32 .
drwxr-xr-x 24 root root 4096 Sep 21 09:32 ..
-rw-r--r--  1 root root 5151 Sep 21 09:32 example_node.py.example
-rw-r--r--  1 root root 1301 Sep 21 09:32 websocket_image_save.py


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server